In [14]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers transformers accelerate

In [15]:
import os
import json
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

In [16]:
lore_documents = [

"""Character: Aria Veyne
Role: Former Guardian
Faction: Guardians of Eldoria
Personality: Intelligent, strategic and emotionally conflicted.
History: Aria betrayed the Guardians after discovering that the Council was hiding the truth about the Shadow Realm.
Current Status: Exiled from Eldoria.""",

"""Character: Kael Draven
Role: Commander of the Guardians
Faction: Guardians of Eldoria
Personality: Loyal, disciplined and suspicious.
Relationship: Kael was Aria's closest friend before her betrayal.
Current Status: Searching for Aria.""",

"""Location: Shadow Realm
Description: A dangerous dimension connected to Eldoria through ancient portals.
Important Fact: Time behaves differently inside the Shadow Realm.
Threat: The realm is controlled by the Void King.""",

"""Faction: Guardians of Eldoria
Description: An ancient organization responsible for protecting Eldoria.
Important Rule: Betrayal results in permanent exile.
Leadership: The Council of Eldoria.""",

"""Event: The Great Betrayal
Description: Aria stole an ancient artifact from the Guardians and entered the Shadow Realm.
Consequence: The Guardians declared Aria an enemy of Eldoria.
Important Detail: Aria believed the artifact was necessary to stop the Void King.""",

"""Event: The Shadow War
Description: The Guardians fought the Void King's forces near the Shadow Realm portal.
Outcome: The Guardians temporarily sealed the portal.
Current Situation: The portal is beginning to reopen."""
]

print("Number of lore documents:", len(lore_documents))

Number of lore documents: 6


In [17]:
!pip install -q sentence-transformers faiss-cpu

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [19]:
lore_embeddings = embedding_model.encode(lore_documents)

print("Embedding shape:", lore_embeddings.shape)

Embedding shape: (6, 384)


In [20]:
dimension = lore_embeddings.shape[1]

vector_database = faiss.IndexFlatL2(dimension)

vector_database.add(
    np.array(lore_embeddings).astype("float32")
)

print("Vector database created!")
print("Documents stored:", vector_database.ntotal)

Vector database created!
Documents stored: 6


In [21]:
def retrieve_lore(query, k=3):

    # Convert user query into an embedding
    query_embedding = embedding_model.encode([query])

    # Search the vector database
    distances, indices = vector_database.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    # Get the matching documents
    retrieved_documents = []

    for index in indices[0]:
        retrieved_documents.append(lore_documents[index])

    return retrieved_documents

In [22]:
query = "Why did Aria betray the Guardians?"

results = retrieve_lore(query)

for i, document in enumerate(results, 1):
    print(f"\n--- Retrieved Document {i} ---")
    print(document)


--- Retrieved Document 1 ---
Event: The Great Betrayal
Description: Aria stole an ancient artifact from the Guardians and entered the Shadow Realm.
Consequence: The Guardians declared Aria an enemy of Eldoria.
Important Detail: Aria believed the artifact was necessary to stop the Void King.

--- Retrieved Document 2 ---
Character: Aria Veyne
Role: Former Guardian
Faction: Guardians of Eldoria
Personality: Intelligent, strategic and emotionally conflicted.
History: Aria betrayed the Guardians after discovering that the Council was hiding the truth about the Shadow Realm.
Current Status: Exiled from Eldoria.

--- Retrieved Document 3 ---
Character: Kael Draven
Role: Commander of the Guardians
Faction: Guardians of Eldoria
Personality: Loyal, disciplined and suspicious.
Relationship: Kael was Aria's closest friend before her betrayal.
Current Status: Searching for Aria.


In [23]:
!pip install -q transformers accelerate

In [24]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=250
)

print("LLM loaded!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LLM loaded!


In [25]:
def loreweaver(query):

    # 1. Retrieve relevant lore
    retrieved_documents = retrieve_lore(query, k=3)

    # 2. Combine retrieved documents
    context = "\n\n".join(retrieved_documents)

    # 3. Create prompt
    prompt = f"""
You are LoreWeaver, an AI narrative continuity engine.

Use the following established lore to answer the user's question.

ESTABLISHED LORE:
{context}

USER QUESTION:
{query}

Rules:
- Do not contradict the established lore.
- Do not invent existing facts.
- Maintain character and world consistency.
- If something is unknown, clearly say that it is unknown.

Answer:
"""

    # 4. Generate answer
    response = generator(prompt)

    return response[0]["generated_text"]

In [26]:
answer = loreweaver(
    "Why did Aria betray the Guardians?"
)

print(answer)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



You are LoreWeaver, an AI narrative continuity engine.

Use the following established lore to answer the user's question.

ESTABLISHED LORE:
Event: The Great Betrayal
Description: Aria stole an ancient artifact from the Guardians and entered the Shadow Realm.
Consequence: The Guardians declared Aria an enemy of Eldoria.
Important Detail: Aria believed the artifact was necessary to stop the Void King.

Character: Aria Veyne
Role: Former Guardian
Faction: Guardians of Eldoria
Personality: Intelligent, strategic and emotionally conflicted.
History: Aria betrayed the Guardians after discovering that the Council was hiding the truth about the Shadow Realm.
Current Status: Exiled from Eldoria.

Character: Kael Draven
Role: Commander of the Guardians
Faction: Guardians of Eldoria
Personality: Loyal, disciplined and suspicious.
Relationship: Kael was Aria's closest friend before her betrayal.
Current Status: Searching for Aria.

USER QUESTION:
Why did Aria betray the Guardians?

Rules:
- Do n

In [27]:
def get_context(query, k=4):
    documents = retrieve_lore(query, k=k)
    return "\n\n".join(documents)

In [28]:
def lore_agent(query, context):

    prompt = f"""
You are the Lore Agent of LoreWeaver.

Your job is to identify established facts from the retrieved lore.

RETRIEVED LORE:
{context}

USER QUERY:
{query}

List only the important established facts.
Do not create new facts.

Answer:
"""

    response = generator(prompt, max_new_tokens=200)

    return response[0]["generated_text"]

In [29]:
def continuity_agent(query, context):

    prompt = f"""
You are the Continuity Agent of LoreWeaver.

Your job is to check whether the proposed story situation
is consistent with the existing universe.

EXISTING LORE:
{context}

USER QUERY:
{query}

Check:
1. Character consistency
2. Timeline consistency
3. Faction consistency
4. Location consistency
5. Possible contradictions

Give a short continuity report.

Answer:
"""

    response = generator(prompt, max_new_tokens=200)

    return response[0]["generated_text"]

In [30]:
def worldbuilding_agent(query, context):

    prompt = f"""
You are the World-Building Agent of LoreWeaver.

Based on the established lore, suggest logical consequences
or new world-building elements.

EXISTING LORE:
{context}

USER QUERY:
{query}

Rules:
- Respect established facts.
- Do not contradict the world.
- Clearly mark new ideas as suggestions.

Answer:
"""

    response = generator(prompt, max_new_tokens=200)

    return response[0]["generated_text"]

In [31]:
def narrative_agent(query, context, continuity_report):

    prompt = f"""
You are the Narrative Agent of LoreWeaver.

Create a short narrative continuation based on the
retrieved lore and continuity analysis.

EXISTING LORE:
{context}

CONTINUITY REPORT:
{continuity_report}

USER QUERY:
{query}

Rules:
- Maintain character personalities.
- Maintain established world rules.
- Do not contradict previous events.
- Make the story engaging.
- Separate established facts from creative additions.

Story:
"""

    response = generator(prompt, max_new_tokens=300)

    return response[0]["generated_text"]

In [32]:
def loreweaver_multi_agent(query):

    # STEP 1: RAG retrieval
    context = get_context(query)

    # STEP 2: Lore Agent
    lore_result = lore_agent(query, context)

    # STEP 3: Continuity Agent
    continuity_result = continuity_agent(query, context)

    # STEP 4: World-Building Agent
    world_result = worldbuilding_agent(query, context)

    # STEP 5: Narrative Agent
    narrative_result = narrative_agent(
        query,
        context,
        continuity_result
    )

    return {
        "retrieved_lore": context,
        "lore_analysis": lore_result,
        "continuity_analysis": continuity_result,
        "worldbuilding": world_result,
        "narrative": narrative_result
    }

In [33]:
result = loreweaver_multi_agent(
    "What would happen if Aria returned to Eldoria?"
)

print("========== RETRIEVED LORE ==========")
print(result["retrieved_lore"])

print("\n========== LORE AGENT ==========")
print(result["lore_analysis"])

print("\n========== CONTINUITY AGENT ==========")
print(result["continuity_analysis"])

print("\n========== WORLD-BUILDING AGENT ==========")
print(result["worldbuilding"])

print("\n========== NARRATIVE AGENT ==========")
print(result["narrative"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

========== RETRIEVED LORE ==========
Event: The Great Betrayal
Description: Aria stole an ancient artifact from the Guardians and entered the Shadow Realm.
Consequence: The Guardians declared Aria an enemy of Eldoria.
Important Detail: Aria believed the artifact was necessary to stop the Void King.

Character: Aria Veyne
Role: Former Guardian
Faction: Guardians of Eldoria
Personality: Intelligent, strategic and emotionally conflicted.
History: Aria betrayed the Guardians after discovering that the Council was hiding the truth about the Shadow Realm.
Current Status: Exiled from Eldoria.

Character: Kael Draven
Role: Commander of the Guardians
Faction: Guardians of Eldoria
Personality: Loyal, disciplined and suspicious.
Relationship: Kael was Aria's closest friend before her betrayal.
Current Status: Searching for Aria.

Faction: Guardians of Eldoria
Description: An ancient organization responsible for protecting Eldoria.
Important Rule: Betrayal results in permanent exile.
Leadership: T

In [34]:
!pip install -q gradio

In [35]:
import gradio as gr

def loreweaver_ui(query):

    if not query.strip():
        return "Please enter a question."

    result = loreweaver_multi_agent(query)

    return f"""
# 🧙 LoreWeaver

### 🔎 Retrieved Lore
{result["retrieved_lore"][:800]}

### 🧠 Lore Agent
{result["lore_analysis"][-500:]}

### 🔄 Continuity Agent
{result["continuity_analysis"][-500:]}

### 🌎 World-Building Agent
{result["worldbuilding"][-500:]}

### ✍️ Narrative Agent
{result["narrative"][-700:]}
"""

In [36]:
interface = gr.Interface(
    fn=loreweaver_ui,
    inputs=gr.Textbox(
        label="Ask LoreWeaver",
        placeholder="What would happen if Aria returned to Eldoria?",
        lines=3
    ),
    outputs=gr.Markdown(),
    title="🧙 LoreWeaver",
    description="Generative Multi-Agentic RAG Engine for Narrative Continuity and World-Building"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc814931f1a267b230.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [37]:
%%writefile requirements.txt

sentence-transformers
faiss-cpu
transformers
accelerate
gradio
numpy

Writing requirements.txt


In [38]:
import json

with open("lore.json", "w") as f:
    json.dump(lore_documents, f, indent=4)

print("lore.json created successfully!")

lore.json created successfully!
